# Lesson 47 · Writing the mel filterbank — building Whisper's ears: the log-mel pipeline, STFT → power spectrum → mel → log in one go

**Goal**: actually build Whisper's front end: STFT → power spectrum → mel → log, ending with `(n_frames, n_mels)`.

> **The soul of the pipeline**: for every step, draw a **shape-change table** (frames×freqs → frames×mels) and you'll never get lost among the dimensions. This is the direct ancestor of the input features in L70.

🔗 Aurora connection: `aurora.audio.mel.mel_spectrogram()` — the input features for the Month 2 keyword-spotting classifier.

← **Previous lesson** · [L46 · The mel scale](L46_mel.ipynb)

> Last lesson covered **the mel scale**: logarithmic pitch perception, mel = 2595·log₁₀(1+f/700), triangular filters.
> This lesson takes **the log-mel pipeline** from formula all the way into code.

## Learning goals

1. Work out the mel energy matrix from the power spectrum by hand: understand how the shapes flow through `power @ fb.T`
2. Explain what eps does numerically: why log(1e-8) ≈ -18.4 is an acceptable floor
3. Implement `log_mel_spectrogram` on your own and match the `aurora.audio.mel` reference output to within 1e-6
4. Explain the nonlinear spread of the mel scale: why the first 40 bins cover 0–1730 Hz and the last 40 cover 1730–8000 Hz


## This lesson's story: why does speech recognition use log-mel instead of the raw waveform?

When you say "play some music" to Siri, what it recognizes isn't the raw PCM numbers the microphone recorded. The raw waveform holds 16,000 samples per second, but 99% of that is phase noise — no help at all for recognition.

The real input to speech recognition is the **log-mel spectrum**: first squeeze the signal onto the frequencies the ear perceives, then compress loudness with a log, and feed one 80-dimensional vector per frame into the neural network. This is the standard input format for Whisper and for most mainstream ASR models.

**The complete 5-step pipeline**:

```
x          (T,)            ← raw audio
↓ STFT
S          (n_frames, n_fft//2+1)   ← complex spectrum
↓ |·|²
power      (n_frames, n_fft//2+1)   ← power spectrum
↓ @ mel_filterbank.T
mel_energy (n_frames, n_mels)       ← mel energy
↓ log(·+ε)
log_mel    (n_frames, n_mels)       ← final features ✓
```

**The core formula** (one line):
$$\text{log\_mel} = \log\!\left(\text{power} \cdot F^T + \varepsilon\right)$$

- `F`: the mel filterbank matrix, shape `(n_mels, n_fft//2+1)`
- `ε = 1e-8`: prevents log(0) = −∞ by clamping the minimum at `log(1e-8) ≈ -18`

Your task in this lesson: implement `log_mel_spectrogram(x, sr, n_mels=80, win_len=1024, hop=256)` so that its output differs from `aurora.audio.mel.mel_spectrogram` by < 1e-6.

## A 2-minute review before we start: matrix multiplication is filtering

- `power @ fb.T` projects each frame's power spectrum onto the mel triangular filters
- The `fb` here is the triangular filterbank from L46
- Remember the shapes: `power (T,F)` × `fb.T (F,M)` → `mel_energy (T,M)`

You already saw the matrix version of the same idea in L21; it's just showing up again in Audio DSP.

### 🔍 Why can't the matrix product `power @ fb.T` be flipped around?

**Many people's first reaction**: "Filtering = multiply the power spectrum by the filter," so why not just write `fb @ power`?

The answer is that **the shapes don't fit** — but the **algorithmic meaning** behind it matters more.

#### The shape view

- `fb` has shape `(n_mels, n_fft//2+1)` — each **row** is one mel filter
- `power` has shape `(n_frames, n_fft//2+1)` — each **row** is one frame's power spectrum
- `fb @ power` = `(n_mels, n_fft//2+1) @ (n_frames, n_fft//2+1)` → **can't be multiplied** (dimensions don't match)
- `power @ fb.T` = `(n_frames, n_fft//2+1) @ (n_fft//2+1, n_mels)` → ✓ gives `(n_frames, n_mels)`

#### The algorithmic view (with a tiny example)

Suppose there's just 1 frame, 5 FFT bins and 2 mel filters:

```
power (one frame) = [p₁, p₂, p₃, p₄, p₅]  shape (1, 5)
fb (2 filters)  = [[f₁₁, f₁₂, f₁₃, f₁₄, f₁₅],    shape (2, 5)
                   [f₂₁, f₂₂, f₂₃, f₂₄, f₂₅]]
```

**Computed with `power @ fb.T`**:

```
power @ fb.T = [p₁, p₂, p₃, p₄, p₅] @ [[f₁₁, f₂₁],
                                       [f₁₂, f₂₂],
                                       [f₁₃, f₂₃],
                                       [f₁₄, f₂₄],
                                       [f₁₅, f₂₅]]

= [p₁·f₁₁ + p₂·f₁₂ + p₃·f₁₃ + p₄·f₁₄ + p₅·f₁₅,
   p₁·f₂₁ + p₂·f₂₂ + p₃·f₂₃ + p₄·f₂₄ + p₅·f₂₅]

= [inner product of mel filter 1 with this frame,
   inner product of mel filter 2 with this frame]
```

**Physical meaning**: each number in `power @ fb.T` = "this frame's power-spectrum vector" · "one mel filter vector" = a **projection**. You're asking "how much of this frame's energy falls in frequency channel m" — which is exactly the definition of filtering.

**If you force `fb @ power`** (pretending it could be multiplied):

```
fb @ power = [[f₁₁, f₁₂, ...], × [p₁, p₂, ..., ...frame 1],
              [f₂₁, f₂₂, ...]]   [p₁, p₂, ..., ...frame 2]
                                 [...]

last row of the result: f₂₁ · [p₁ of frame 1] + f₂₂ · [p₂ of frame 1] + ...
```

This mixes up the roles of "filter" and "frame": you'd be multiplying each number of a filter by data from different frames, which has no physical meaning.

So **`power @ fb.T` is the only correct order**.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine, chirp

sr = 16000
# Make a test signal: a 2-second chirp (200 Hz → 4000 Hz)
x = chirp(200, 4000, duration=2.0, sample_rate=sr)
print(f"x.shape = {x.shape}, sr = {sr}")

## 1. What is log-mel?

The core formula is just one line:

```
log_mel = log( power_spectrum @ mel_filterbank.T + eps )
```

where:
- `mel_filterbank` has shape `(n_mels, n_fft//2+1)`: each row is the weight vector of one triangular band-pass filter
- `power_spectrum` has shape `(n_frames, n_fft//2+1)`: each frame's power spectrum (|STFT|²)
- `eps = 1e-8`: prevents log(0) — numerical protection
- the result has shape `(n_frames, n_mels)`

**Why the log compression**: our perception of loudness is logarithmic (going from 1 to 2 units of energy feels like a far bigger step than going from 1,000,000 to 1,000,001). Without the log, mel energies span many orders of magnitude and a neural network struggles to learn from them; after log compression the values are spread more evenly and the gradients are more stable.

---

### Step one: understand the mel-scale formula

**Definition of the mel scale** (measured experimentally by Stanley Smith Stevens in 1937):

$$\text{mel} = 2595 \cdot \log_{10}\left(1 + \frac{f}{700}\right)$$

**Where does this formula come from?**

1. **The ear hears nonlinearly**: trained listeners took part in "pitch-matching" experiments — they were given a reference frequency (say 1000 Hz) and a test tone, and adjusted the test tone's frequency until its pitch stood in the requested relation to the reference. The finding:
   - 1000 Hz is defined as 1000 mel (an agreed reference point)
   - the "height" of 2000 Hz is only about 1500 mel (not 2000 mel!)
   - the "height" of 200 Hz is about 300 mel

   That means high frequencies get "squashed": in perceived pitch, the jump from 100 to 200 Hz feels far bigger than the jump from 4000 to 4100 Hz.

2. **Where the log comes from**: the experimental data are fitted with $\log_{10}(1 + f/700)$, and the factor 2595 is chosen so that 1000 Hz = 1000 mel. This **isn't derived** — it's **the result of measurement plus curve fitting**.

   - 700 Hz: a characteristic frequency of human hearing (it sets where the scale bends)
   - 2595: chosen so that mel(1000 Hz) = 1000

**What you need to remember**: mel scale = logarithmic scale = the ear's pitch-perception scale. Points spread evenly on the mel scale automatically become uneven when mapped back to Hz (dense at low frequencies, sparse at high ones).

---

### Step two: why triangular filters and not rectangles?

**Two physical facts about the ear decide it**:

#### Fact 1: critical bandwidth and masking (an everyday example)

Imagine you're in a noisy café. A friend is talking right by your ear (2000 Hz), while the background music has a noise component at 2050 Hz. Can you make out what your friend says?

**The answer is usually "not clearly."** Although 2000 Hz and 2050 Hz differ by only 50 Hz, the two sounds "blend together" and your brain can't separate them. This is called **masking**.

Behind this lies **the physical structure of the cochlea's basilar membrane**: the hair cells (the receptors) on the basilar membrane aren't laid out as discrete, separate units — their responses **overlap by about 50%**. The group of hair cells excited by a loud tone at one frequency partly "suppresses" weak signals at nearby frequencies.

**Expressing this fact with filters**:
- With rectangular filters (weight 1 everywhere from 0 to 1000 Hz, 0 everywhere from 1000 to 2000 Hz), neighboring frequency channels are completely independent and can't reflect this natural overlap.
- With triangular filters, two neighboring filters **overlap by about 50%** (they cross at a height of 0.5, halfway between their centers; each triangle reaches zero exactly at its neighbor's peak), which mirrors the physical overlap in the cochlea.

#### Fact 2: smooth transitions, no hard cuts

The two sloping sides of a triangle (rising and falling) give **smooth transitions in frequency**. That avoids a "Gibbs phenomenon" at filter edges — a rectangular filter jumps sharply at its cutoff, causing spectral leakage, while the gradual slope of a triangle fits the way natural signals fall off in frequency.

#### Conclusion: the parameters of the mel triangular filters

- **Position**: set by even spacing on the mel scale (take n_mels+2 evenly spaced points in the mel domain from 0 to hz_to_mel(sr/2), then convert back to Hz)
- **Shape**: a triangle with its peak at the center frequency (one point in the Hz domain), falling linearly to 0 on both sides
- **Overlap**: neighboring filters overlap by 50% (implemented by having the right slope of triangle m and the left slope of triangle m+1 share the same span)
- **Area**: Aurora follows the **HTK convention** — the peak is 1 (the Slaney-style area scaling 2/(f_right - f_left) is **not** applied), so there is **no area normalization**. As a result the wide triangles at high frequencies have much more area than the narrow ones at low frequencies, which means high-frequency energy is automatically boosted (see the detailed derivation in L46).

In [ ]:
# Demo: log compression evens out the energy distribution
from aurora.audio.mel import mel_filterbank, mel_spectrogram
from aurora.audio.stft import stft

S = stft(x, n_fft=1024, hop_length=256)
power = np.abs(S) ** 2
fb = mel_filterbank(80, 1024, sr)
mel_energy = power @ fb.T          # linear mel energy

print(f"linear mel energy range: [{mel_energy.min():.4f}, {mel_energy.max():.4f}]")
print(f"log    mel energy range: [{np.log(mel_energy + 1e-8).min():.4f}, {np.log(mel_energy + 1e-8).max():.4f}]")

## 2. Shape flow and the spread of mel bins

The shape change at each step:

```
x          : (T,)                        # raw waveform, T samples
STFT       : (n_frames, n_fft//2 + 1)   # complex matrix
power      : (n_frames, n_fft//2 + 1)   # |STFT|², real
fb         : (n_mels,   n_fft//2 + 1)   # mel filterbank
mel_energy : (n_frames, n_mels)          # power @ fb.T
log_mel    : (n_frames, n_mels)          # log(mel_energy + eps)
```

The key transpose: `power @ fb.T`, not `fb @ power`. Each row of `fb` is one mel filter (length = number of FFT bins), each row of `power` is one frame's power spectrum, and after the matrix product each output row holds the n_mels energy values for one frame.

---

### The nonlinear spread of mel bins

**A common question**: "Why do the first 40 mel bins cover 0–1730 Hz while the last 40 cover 1730–8000 Hz?"

The answer is in **the definition of the mel scale** — it looks unfair, but it matches the ear exactly.

#### The derivation

1. **Split the mel domain evenly** (this is the key):
   - Total range: from 0 to hz_to_mel(8000) in the mel domain
   - With n_mels = 80 we need 80 + 2 = 82 edge points (one extra at each end to fix the filters' lower and upper bounds)
   - Take 82 evenly spaced points in the mel domain: mel_points = np.linspace(0, hz_to_mel(8000), 82)

2. **Convert back to Hz** (this is the nonlinear mapping):
   ```python
   hz_points = mel_to_hz(mel_points)
   # mel_to_hz is the inverse of the mel formula: f = 700 * (10^(mel/2595) - 1)
   ```

3. **The result is uneven in Hz**:
   - mel_points[0:41] correspond to 0–1730 Hz (40 bins, i.e. 41 edges)
   - mel_points[41:82] correspond to 1730–8000 Hz (40 bins)

#### Why is this right?

Because **the mel scale is itself a logarithmic scale**. An even spread on a log scale automatically becomes "dense at low frequencies, sparse at high ones" when mapped back to a linear scale.

**Physical meaning**: low frequencies (< 2 kHz) are the key region for speech (vowel formants) and need more frequency resolution; high frequencies (> 2 kHz) matter less for telling vowels apart, so coarse resolution is enough. The mel scale encodes exactly this priority.

#### Verification code
```python
from aurora.audio.mel import hz_to_mel, mel_to_hz

# the 82 edge points of the 80 mel bins
mel_pts = np.linspace(0, hz_to_mel(8000), 82)

# frequency at point 40
hz_at_40 = mel_to_hz(mel_pts[40])
print(f"around the middle of the 80 bins (point 40): {hz_at_40:.0f} Hz")  # ≈ 1730 Hz

# check
print(f"first 40 bins: {mel_to_hz(mel_pts[0]):.0f} – {hz_at_40:.0f} Hz")
print(f"last 40 bins: {hz_at_40:.0f} – {mel_to_hz(mel_pts[81]):.0f} Hz")
```

In [ ]:
# Check the shape at every step
print(f"x        : {x.shape}")
print(f"STFT     : {S.shape}")
print(f"power    : {power.shape}")
print(f"fb       : {fb.shape}")
print(f"mel_enrg : {mel_energy.shape}")
print(f"log_mel  : {np.log(mel_energy + 1e-8).shape}")

# Plot the first 5 filters of the mel filterbank
freqs = np.fft.rfftfreq(1024, 1/sr)
fig, ax = plt.subplots(figsize=(9, 3))
for i in range(5):
    ax.plot(freqs, fb[i], label=f"Mel#{i}")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Weight")
ax.set_title("Mel filterbank (first 5 triangular filters)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 3. eps and mean normalization

**eps = 1e-8**: the one and only numerical guard. log(0) = -∞ would make the gradients blow up; adding eps clamps the minimum at `log(1e-8) ≈ -18.4`, which physically corresponds to extremely faint energy and doesn't affect the frames that matter.

**Computing log(1e-8)** (natural log, ln):

$$\ln(10^{-8}) = -8 \times \ln(10) = -8 \times 2.303 \approx -18.42$$

In other words, even when mel_energy is close to 0, adding eps turns it into 1e-8, and the log presses it down to a safe floor of about -18.4. That floor is far from -∞ (so it won't cause gradient explosion), yet small enough (so the network won't learn the false idea that "zero energy carries information").

**Mean normalization (CMVN)**: before training, the whole clip is usually normalized:

```
log_mel_norm = (log_mel - log_mel.mean(axis=0)) / (log_mel.std(axis=0) + 1e-8)
```

`axis=0` normalizes each mel bin independently, removing bias from the microphone's frequency response and making the model more stable across recording conditions. The `1e-8` in the denominator also guards against dividing by zero if some bin has zero variance (rare, but a numerical guard never hurts).

The Month 2 classifier's preprocessing layer does this step; `mel_spectrogram` itself leaves it out, so it keeps doing one job only.

In [ ]:
# Demo: the effect of CMVN normalization
log_mel = np.log(mel_energy + 1e-8)
log_mel_norm = (log_mel - log_mel.mean(axis=0)) / (log_mel.std(axis=0) + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(log_mel.T, aspect="auto", origin="lower", cmap="magma")
axes[0].set_title("log-mel (raw)")
axes[0].set_xlabel("Frame"); axes[0].set_ylabel("Mel bin")

axes[1].imshow(log_mel_norm.T, aspect="auto", origin="lower", cmap="magma")
axes[1].set_title("log-mel (after CMVN)")
axes[1].set_xlabel("Frame"); axes[1].set_ylabel("Mel bin")
plt.tight_layout()
plt.show()

print(f"before normalization: mean={log_mel.mean():.3f}  std={log_mel.std():.3f}")
print(f"after normalization:  mean={log_mel_norm.mean():.3f}  std={log_mel_norm.std():.3f}")

### 3.1 An aside: where does the `atol=1e-6` in the acceptance test come from?

*(Ruth's question: the check code in section 4.1 below uses `np.allclose(out, ref, atol=1e-6)` to decide whether your implementation is right — is that 1e-6 picked at random, or is there a reason?)*

**Start with an analogy.** Compute 1/3 on a calculator and the screen shows `0.3333333333`, but the true 1/3 is an infinitely repeating decimal — the calculator can only store a finite number of digits, so the last digit is bound to be "made to fit." Floating-point numbers in a computer (float64) work the same way: they store a real number in just 64 bits, so only finitely many numbers can be represented exactly, and any result that lands between two "representable numbers" gets rounded to the nearest one.

**Just how small is this "rounding error"?** NumPy can tell you directly:

```python
np.finfo(np.float64).eps  # ≈ 2.22e-16
```

This number is called **machine epsilon**: the relative gap between two neighboring numbers that float64 can represent exactly, about 2.22 × 10⁻¹⁶. So every **single** floating-point operation (an add, a multiply, …) has a relative error around this size — small enough to ignore.

**But this pipeline doesn't do just one operation.** `log_mel_spectrogram` goes through STFT → magnitude → square → matrix product → log, five steps in all, and each one adds a little fresh rounding error; the errors snowball. Worse is the last step: the `log` function is especially sensitive to inputs near 0 — its derivative is `1/x`, so the smaller `x` is, the more a tiny input error gets amplified. A concrete number: if some value in `mel_energy` happens to sit near `eps = 1e-8`, then even if that value's relative error is as small as 1e-13, after the `log` the absolute error can be amplified to around `1e-13 × (1/1e-8) = 1e-5` — 10 orders of magnitude bigger than float64 machine epsilon!

Stack these effects together — the accumulated error of five steps plus the amplifying effect of `log` on small values — and the difference between a **perfectly correct** implementation and the reference usually lands in the range **1e-10 to 1e-6**, not at machine epsilon's 1e-16. `atol=1e-6` sits at the top of that range to leave a safety margin for two correct implementations that "take different computational paths but are mathematically equivalent."

**So why not loosen it to 1e-4 to be extra safe?** Because the tolerance also has to catch **real bugs**. If you skip the squaring step (using `|S|` instead of `|S|²`), or get the filter matrix transpose backwards, the error is typically on the order of `O(0.1) to O(10)` — several orders of magnitude bigger than 1e-6, so `allclose` catches it instantly. 1e-6 sits in the middle ground between "tolerating the path differences of correct implementations" and "exposing wrong ones."

**Back to Ruth's specific follow-up**: "If I use a slightly different numerical order (say, adding a constant after the log vs before it), will the error go past 1e-6?" — First we need to separate two things:

1. `log(x) + c` and `log(x + c)`: these two formulas simply aren't the same thing mathematically (unless c = 0). That's not "a different order" — it's **the wrong formula**. Their difference is huge when x is small, nothing like the size of rounding error, and `atol=1e-6` catches it easily — this isn't the "numerical error" we're talking about; it's an implementation bug.
2. A genuine "different order of operations," such as computing the matrix product frame by frame and stacking the results vs doing it all at once — these are strictly equivalent mathematically, and only the intermediate floating-point steps differ (addition isn't actually associative in floating point). The difference they produce is the "rounding error" we mean, roughly 1e-13 to 1e-9, far below the 1e-6 tolerance.

The code below puts the sizes of these two kinds of "error" side by side, and you'll see they're several orders of magnitude apart — which is why 1e-6 is both lenient and strict: lenient enough to admit every correct implementation that "reaches the same place by a different road," strict enough that no real bug can hide.

**One more reference point**: the common audio libraries (librosa, torchaudio) also generally test their own mel-spectrogram implementations with tolerances on the order of 1e-5 to 1e-6, not something close to machine epsilon like 1e-12 — for exactly the same reason: the FFT + log combination naturally needs room for implementations that take different paths to an equivalent result.

In [ ]:
# Demo: how many orders of magnitude apart are the two kinds of "error"?

eps_machine = np.finfo(np.float64).eps
print(f"float64 machine epsilon eps ≈ {eps_machine:.2e}\n")

# ── Scenario A: mathematically identical, only the order of operations differs (this is real "rounding error") ──
# All at once vs frame by frame then stacked — in theory the results should be exactly the same
mel_a = power @ fb.T
mel_b = np.stack([power[i] @ fb.T for i in range(power.shape[0])])
err_order = np.max(np.abs(mel_a - mel_b))
print(f"[different order] power @ fb.T all at once vs frame by frame → max error = {err_order:.2e}")
print("  → this is genuine floating-point rounding error, far below atol=1e-6\n")

# ── Scenario B: the wrong formula (not rounding error — a bug) ──
# Use a few "faint energy" bins near eps=1e-8 — exactly the case the eps guard is there for
x_small = np.array([1e-10, 1e-9, 1e-8, 1e-7, 1e-6])
formula_correct = np.log(x_small + 1e-8)   # correct formula: add eps, then log
formula_wrong = np.log(x_small) + 1e-8     # wrong intuition: log first, then add eps
err_formula = np.max(np.abs(formula_correct - formula_wrong))
print(f"[wrong formula]   log(x+eps) vs log(x)+eps                  → max error = {err_formula:.2e}")
print("  → this isn't rounding error; they're two mathematically different formulas, far beyond 1e-6\n")

print(f"the two differ by a factor of ≈ {err_formula / max(err_order, 1e-300):.1e} —")
print("which is why atol=1e-6 lets 'different order' through but catches 'wrong formula'.")

## 4. ✏️ Implement `log_mel_spectrogram(x, sr, n_mels=80, win_len=1024, hop=256)`

**Four-step hints**:

1. **Step one: call the STFT**
   ```python
   S = stft(x, n_fft=win_len, hop_length=hop)
   # S is a complex matrix, shape (n_frames, win_len//2+1)
   ```

2. **Step two: compute the power spectrum**
   ```python
   magnitude = np.abs(S)      # turn complex numbers into magnitudes (real)
   power = magnitude ** 2     # square
   # power has shape (n_frames, win_len//2+1)
   ```

3. **Step three: get the mel filterbank and apply it**
   ```python
   fb = mel_filterbank(n_mels, win_len, sr)
   # fb has shape (n_mels, win_len//2+1)
   mel_energy = power @ fb.T  # matrix product
   # mel_energy has shape (n_frames, n_mels)
   ```

4. **Step four: log compression (add eps to avoid log(0))**
   ```python
   return np.log(mel_energy + 1e-8)
   # returns shape (n_frames, n_mels)
   ```

**If you're stuck, go back to**: L44 (STFT review) / L46 (mel filter review)

**Reference implementation**: `solutions/L47_mel_implement_solutions.md`

## 4.1 Signatures and usage of the key functions

Before you implement `log_mel_spectrogram`, get clear on the two functions you depend on.

### `stft(x, n_fft=..., hop_length=...)`

**Import**:
```python
from aurora.audio.stft import stft
```

**Signature**:
```python
def stft(x, n_fft=1024, hop_length=256, window='hann'):
    """
    x        : 1-D float array, the raw waveform
    n_fft    : FFT window length, default 1024
    hop_length : hop between frames (in samples), default 256
    window   : window function name, default 'hann' (Hann window)

    returns  : (n_frames, n_fft//2 + 1) complex array (the STFT)
    """
```

**Key points**:
- **It returns a complex matrix.** Each number has the form `a + b*j`, with magnitude `|a + b*j| = sqrt(a² + b²)`
- `n_fft` is the window length, often also called `win_len`
- `hop_length` is the hop between frames, written `hop` in this lesson

**Example**:
```python
S = stft(x, n_fft=1024, hop_length=256)  # S.shape = (some n_frames, 513)
# 513 = 1024 // 2 + 1 (length of the FFT's real-signal spectrum)

# get the magnitude (the step before the power spectrum)
magnitude = np.abs(S)       # magnitude.shape = (n_frames, 513), real
power = magnitude ** 2      # power.shape = (n_frames, 513), real
```

### `mel_filterbank(n_mels, n_fft, sample_rate)`

**Import**:
```python
from aurora.audio.mel import mel_filterbank
```

**Signature**:
```python
def mel_filterbank(n_mels, n_fft, sample_rate):
    """
    n_mels      : number of mel filters, usually 80
    n_fft       : FFT window length (must match the STFT's n_fft!)
    sample_rate : sample rate, usually 16000 Hz

    returns     : (n_mels, n_fft//2 + 1) real array
                  each row is the weight vector of one triangular filter
    """
```

**Key points**:
- **n_fft must match the stft's n_fft**, or the dimensions won't line up
- It returns a real matrix; each row can be used directly for filtering
- The argument order is `(n_mels, n_fft, sample_rate)`

**Example**:
```python
fb = mel_filterbank(80, 1024, 16000)  # fb.shape = (80, 513)
# each row is one mel filter; all the rows together (no transpose) roughly cover the whole frequency axis

# check: see what a filter looks like
import matplotlib.pyplot as plt
freqs = np.fft.rfftfreq(1024, 1/16000)
plt.plot(freqs, fb[40])  # plot filter 40
plt.xlabel('Frequency (Hz)')
plt.ylabel('Weight')
plt.show()
```

### Why `n_fft//2 + 1`?

The STFT only outputs the **positive-frequency** part (the negative frequencies mirror the positive ones, and a real signal doesn't need them). For `n_fft=1024`:
- The full FFT output has 1024 complex numbers
- 1 of them is DC (0 Hz), 511 are positive frequencies, 1 is the Nyquist bin (512), and the remaining 511 are negative frequencies (mirrors)
- What's kept is 0 Hz + 511 positive frequencies + Nyquist = 513 = 1024//2 + 1

That's also why the second dimension of the mel filterbank is always `n_fft//2 + 1`.

In [ ]:
# Note: the TODO below uses stft and mel_filterbank (stft was imported earlier)
from aurora.audio.mel import mel_filterbank

def log_mel_spectrogram(x, sr, n_mels=80, win_len=1024, hop=256):
    """
    x       : 1-D numpy array, the raw waveform
    sr      : sample rate (Hz)
    n_mels  : number of mel filters, default 80
    win_len : FFT window length (= n_fft), default 1024
    hop     : frame hop (samples), default 256
    returns : (n_frames, n_mels) float64 log-mel matrix
    """
    # ✏️ TODO step 1: STFT → complex matrix S, shape (n_frames, win_len//2+1)

    # ✏️ TODO step 2: power = |S|², same shape

    # ✏️ TODO step 3: mel_filterbank → fb, shape (n_mels, win_len//2+1)

    # ✏️ TODO step 4: mel_energy = power @ fb.T, shape (n_frames, n_mels)

    # ✏️ TODO step 5: return np.log(mel_energy + 1e-8)
    raise NotImplementedError("TODO: return np.log(power @ fb.T + 1e-8)")

In [ ]:
from aurora.audio.mel import mel_spectrogram

# Build the reference output: aurora's mel_spectrogram has no log, so add log+eps by hand for the reference
ref = np.log(mel_spectrogram(x, sr, n_fft=1024, hop_length=256, n_mels=80) + 1e-8)
out = log_mel_spectrogram(x, sr, n_mels=80, win_len=1024, hop=256)

assert out is not None, "the function returned None — check that it has a return statement"
assert out.shape == ref.shape, f"shape mismatch: {out.shape} vs {ref.shape}"
assert np.allclose(out, ref, atol=1e-6), (
    f"values don't match, max error = {np.max(np.abs(out - ref)):.2e}"
)
print(f"✅ shape = {out.shape}")
print(f"✅ max error = {np.max(np.abs(out - ref)):.2e} (< 1e-6)")
print(f"✅ value range [{out.min():.2f}, {out.max():.2f}]")

## 5. Parameter experiment: n_mels = 40 / 80 / 128

**What to expect**:
- `n_mels=40`: coarse frequency resolution, fewer pixels vertically in the heatmap, low-frequency detail lost, but fast to compute; fine for a small Month 2 model
- `n_mels=80`: the balanced choice and the industry standard (Whisper uses 80)
- `n_mels=128`: more high-frequency detail, good for music/environmental-sound classification, but the feature dimension grows and the model's parameter count grows with it

In the Month 2 experiments you can compare the keyword-spotting val accuracy under the three settings; 80 and 128 usually differ very little.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, n_mels in zip(axes, [40, 80, 128]):
    try:
        feat = log_mel_spectrogram(x, sr, n_mels=n_mels, win_len=1024, hop=256)
    except (NotImplementedError, TypeError):
        feat = None
    if feat is None:  # skip while the stub is unfinished (raises NotImplementedError or returns None)
        ax.set_title(f"n_mels={n_mels}\n(finish the log_mel_spectrogram TODO above first)")
        continue
    im = ax.imshow(feat.T, aspect="auto", origin="lower", cmap="magma")
    ax.set_title(f"n_mels = {n_mels}\nshape {feat.shape}")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Mel bin")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("log-mel spectrogram: comparing n_mels (chirp 200→4000 Hz, 2 s)", y=1.02)
plt.tight_layout()
plt.show()

## Wrap-up

In this lesson you implemented `log_mel_spectrogram(x, sr, n_mels, win_len, hop)`, which outputs an `(n_frames, n_mels)` float64 matrix and matches `aurora.audio.mel.mel_spectrogram` to 1e-6. Inside, the function relies on `aurora.audio.stft.stft` and `aurora.audio.mel.mel_filterbank`, so this step isn't "building yet another approximate version" — it's bolting the whole front-end chain together properly. Next lesson: **L48**, time–frequency plots compared, where several sets of plots make the differences between the linear spectrogram, the mel spectrogram and the log-mel spectrogram really visible.

## ✏️ Closed-book derivation check cell — mel filter normalization

**Rule: close every cell above and do the derivation below from memory alone.**

**Problem**: the triangular filter $H_m(k)$ is defined as:

$$H_m(k) = \begin{cases}
\frac{k - f(m-1)}{f(m) - f(m-1)} & f(m-1) \le k < f(m) \\
\frac{f(m+1) - k}{f(m+1) - f(m)} & f(m) \le k \le f(m+1) \\
0 & \text{otherwise}
\end{cases}$$

**What the piecewise function means geometrically**:

- `f(m-1), f(m), f(m+1)` are three frequency points (in Hz or as FFT bin indices)
- First piece (rising): from `f(m-1)` to `f(m)`, the weight climbs linearly from 0 to 1
  - the numerator `k - f(m-1)` is the distance from the current position to the left end
  - the denominator `f(m) - f(m-1)` is the total width, which normalizes the peak to 1
  - e.g. with k at the midpoint, the weight is 0.5
- Second piece (falling): from `f(m)` to `f(m+1)`, the weight drops linearly from 1 to 0
  - the numerator `f(m+1) - k` is the distance from the current position to the right end
  - the denominator `f(m+1) - f(m)` is the total width
  - e.g. with k at the midpoint, the weight is 0.5
- Everywhere else: the weight is 0 (this filter does nothing there)

**Picture it**: a triangular filter looks like a tent, with `f(m)` as the peak (height 1) and a slope on each side.

---

**Derivation problems**:

1. Write an analytic expression for the filter's area (the sum over all $k$)

   **Hint**: area = $\sum_{k=f(m-1)}^{f(m+1)} H_m(k)$, which you can split into the sum of two triangle areas

2. Does Aurora's implementation normalize the area? If so, write the normalized expression

(Write your derivation here...)

In [ ]:
# Check: does each row sum (area) of the filter matrix match expectations?
import numpy as np
from aurora.audio.mel import mel_filterbank

fb = mel_filterbank(n_mels=40, n_fft=1024, sample_rate=16000)  # shape (40, 513)
row_sums = fb.sum(axis=1)

# Answer to the derivation: Aurora follows the HTK convention — triangle peak ≈ 1, with **no** area normalization.
# So the wider the filter (the higher the frequency), the bigger the row area: narrow low-frequency triangles have small area, wide high-frequency ones large area.
# If you need Slaney-style area normalization, divide each row by its Hz bandwidth yourself.
assert row_sums.min() > 0, "found an all-zero row — the filterbank was built wrong"
assert row_sums[-1] > row_sums[0] * 5, "high-frequency rows should have far more area than low-frequency rows (HTK doesn't normalize area)"
print(f"filter row areas: min={row_sums.min():.4f} (narrow low-freq triangle) → max={row_sums.max():.4f} (wide high-freq triangle)")
print(f"area ratio max/min = {row_sums.max()/row_sums.min():.1f}×")
print("✅ Conclusion: Aurora does no area normalization (HTK peak ≈ 1 convention); row area grows with bandwidth")

In [ ]:
# ✏️ Self-review for this lesson
l47_review = {
    "pipeline_5steps":         None,  # remember the five-step flow STFT→power→@fb.T→log+ε? True/False
    "log_mel_implemented":     None,  # log_mel_spectrogram implemented and passes atol=1e-6? True/False
    "eps_role":                None,  # understand ε=1e-8 prevents log(0)=-inf, minimum ≈ -18? True/False
    "shape_flow":              None,  # can work out the shapes (T,)→(n_frames,n_bins)→(n_frames,n_mels) by hand? True/False
    "whiteboard_passed":       None,  # closed-book whiteboard derivation of filter normalization passed? True/False
}

unfilled = [k for k, v in l47_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l47_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L47 fully cleared! On to L48: time–frequency plots compared (visual lesson)')

---

→ **Next lesson** · [L48 · Time–frequency plots compared](L48_visual_stft.ipynb)

> Next lesson covers **time–frequency plots compared**: a side-by-side visual comparison of the linear, mel and log-mel spectrograms.